In [45]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
import joblib

In [6]:
df = pd.read_csv("data.csv")
print(df.head())

  timestamp  temperature  pressure  vibration   rpm
0  18:39:22           70        38       0.32  1173
1  18:39:27           33        27       0.44  1743
2  18:39:32           31        40       0.43  1122
3  18:39:37           59        40       0.13  1429
4  18:39:42           52        36       0.25  1650


In [28]:
# df['timestamp'] = pd.to_datetime(df['timestamp'])
# df['hour'] = df['timestamp'].dt.hour
# df['minute'] = df['timestamp'].dt.minute
# df['second'] = df['timestamp'].dt.second
# print(df)

In [29]:
X = df[['temperature', 'vibration', 'rpm', 'pressure']]


# Train on earlier data, test on recent data
split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

Train size: 400, Test size: 100


In [30]:
# Create model
model = IsolationForest(
    n_estimators=100,     # number of trees
    contamination=0.02,   # % of anomalies
    random_state=42
)

# Train model
model.fit(X_train)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.02
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [40]:
y_pred = model.predict(X_test)

# Convert labels
labels = ["Normal" if x == 1 else "Anomaly" for x in y_pred]

print(labels[:10])

['Normal', 'Normal', 'Anomaly', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal']


In [41]:
# Check prediction distribution
print(f"Anomalies detected: {sum(y_pred == -1)} / {len(y_pred)}")
print(f"Anomaly rate: {sum(y_pred == -1) / len(y_pred) * 100:.2f}%")

# Statistical check - do anomalies differ from normal?
normal_mean = X_test[y_pred == 1].mean()
anomaly_mean = X_test[y_pred == -1].mean()
print(f"\nNormal avg temperature: {normal_mean['temperature']:.2f}")
print(f"Anomaly avg temperature: {anomaly_mean['temperature']:.2f}")

Anomalies detected: 2 / 100
Anomaly rate: 2.00%

Normal avg temperature: 51.46
Anomaly avg temperature: 51.50


In [38]:
df_test = X_test.copy()

df_test['prediction'] = y_pred
df_test['label'] = labels

In [39]:
print(df_test['label'].value_counts())

label
Normal     98
Anomaly     2
Name: count, dtype: int64


In [42]:
# Step 7: Create results dataframe
df_test = X_test.copy()

df_test['prediction'] = y_pred
df_test['label'] = labels

# Display first few rows
print("First 10 predictions:")
print(df_test.head(10))
print("\nDataframe shape:", df_test.shape)
print("\nColumn names:", df_test.columns.tolist())

First 10 predictions:
     temperature  vibration   rpm  pressure  prediction    label
400           34       0.13  1720        32           1   Normal
401           51       0.44  1486        28           1   Normal
402           69       0.46  1110        22          -1  Anomaly
403           44       0.30  1280        35           1   Normal
404           31       0.10  1368        28           1   Normal
405           47       0.50  1091        23           1   Normal
406           65       0.36  1314        21           1   Normal
407           65       0.35  1911        21           1   Normal
408           68       0.22  1462        29           1   Normal
409           65       0.28  1243        24           1   Normal

Dataframe shape: (100, 6)

Column names: ['temperature', 'vibration', 'rpm', 'pressure', 'prediction', 'label']


In [43]:
# Step 8: Display final results
print("=" * 50)
print("ANOMALY DETECTION RESULTS")
print("=" * 50)

# Summary statistics
print(f"\nTotal test samples: {len(df_test)}")
print(f"Normal readings: {sum(df_test['label'] == 'Normal')}")
print(f"Anomalies detected: {sum(df_test['label'] == 'Anomaly')}")
print(f"Anomaly percentage: {sum(df_test['label'] == 'Anomaly') / len(df_test) * 100:.2f}%")

# Show anomalies
print("\n" + "=" * 50)
print("ANOMALIES DETECTED:")
print("=" * 50)
anomalies = df_test[df_test['label'] == 'Anomaly']
print(anomalies.head(10))

# Save results
df_test.to_csv('anomaly_results.csv', index=False)
print("\n✓ Results saved to 'anomaly_results.csv'")

ANOMALY DETECTION RESULTS

Total test samples: 100
Normal readings: 98
Anomalies detected: 2
Anomaly percentage: 2.00%

ANOMALIES DETECTED:
     temperature  vibration   rpm  pressure  prediction    label
402           69       0.46  1110        22          -1  Anomaly
448           34       0.29  1044        40          -1  Anomaly

✓ Results saved to 'anomaly_results.csv'


In [44]:
# Convert to JSON for API response
results_json = df_test.to_json(orient='records')
print(results_json)

[{"temperature":34,"vibration":0.13,"rpm":1720,"pressure":32,"prediction":1,"label":"Normal"},{"temperature":51,"vibration":0.44,"rpm":1486,"pressure":28,"prediction":1,"label":"Normal"},{"temperature":69,"vibration":0.46,"rpm":1110,"pressure":22,"prediction":-1,"label":"Anomaly"},{"temperature":44,"vibration":0.3,"rpm":1280,"pressure":35,"prediction":1,"label":"Normal"},{"temperature":31,"vibration":0.1,"rpm":1368,"pressure":28,"prediction":1,"label":"Normal"},{"temperature":47,"vibration":0.5,"rpm":1091,"pressure":23,"prediction":1,"label":"Normal"},{"temperature":65,"vibration":0.36,"rpm":1314,"pressure":21,"prediction":1,"label":"Normal"},{"temperature":65,"vibration":0.35,"rpm":1911,"pressure":21,"prediction":1,"label":"Normal"},{"temperature":68,"vibration":0.22,"rpm":1462,"pressure":29,"prediction":1,"label":"Normal"},{"temperature":65,"vibration":0.28,"rpm":1243,"pressure":24,"prediction":1,"label":"Normal"},{"temperature":31,"vibration":0.18,"rpm":1786,"pressure":37,"predictio

In [46]:
joblib.dump(model, "model.pkl")

['model.pkl']